# 04b · POST HOC diagnostic for D0 (information only; does not change any verdict)

*Added after seeing D0 fail (median F1 0.78 < 0.95) in notebook 04.* D0 fed the classifier "planted" vectors: each
pixel's official class as a one-hot vector **plus noise (standard deviation 0.3)** and 16 pure-noise numbers. Notebook
03 measured that these planted maps match the official maps with a median class F1 of only 0.88, so the failure may
come from the noise level chosen for the planted vectors rather than from the classifier-plus-executor pipeline.

This notebook measures that: the same pipeline with **noise-free** planted vectors (one-hot only). If the reference
plans then reproduce the truth, the pipeline itself is sound and D0's failure is a property of the planted input.
The D0 verdict stays FAIL as fixed in `PLAN.md`.

In [1]:
import json
import time

import numpy as np
import pandas as pd

from askmap.catalogue import area_classes
from askmap.config import PROJECT_ROOT, data_root, load_config
from askmap.layers import SOLAR_LATER, official_leaf_map
from askmap.plan import AreaContext, execute
from askmap.power import add_derived, fetch_daily, pixel_cells
from askmap.score import f1
from askmap.tiles import area_tiles, pixel_lonlat
from askmap.universe import Universe, box_of

cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
Q = load_config(PROJECT_ROOT / "configs" / "queries.yaml")["queries"]
RW = load_config(PROJECT_ROOT / "configs" / "reworded.yaml")["reworded"]
ROOT = data_root()
RES = PROJECT_ROOT / "outputs" / "results"
FIG = PROJECT_ROOT / "outputs" / "figures"
CL, P, SC = cfg["classes"], cfg["power"], cfg["scoring"]
LEAVES, GROUPS = CL["leaves"], CL["groups"]
CATALOGUE = area_classes(cfg)
T_START = time.time()
cells = pd.read_parquet(ROOT / "power" / "universe_cells.parquet")
daily = {
    box: add_derived(fetch_daily(P, b, ROOT / "power" / f"universe_{box}_daily.nc"), P["low_wind_sun_pct"])
    for box, b in P["boxes"].items()
}
uni = Universe(cells=cells, daily=daily)
SEEN = ["north_dakota", "north_carolina"]
HELD = cfg["heldout_pools"]  # seen area -> held-out state
POOL_Q = {
    "north_dakota": "north_dakota",
    "north_carolina": "north_carolina",
    "minnesota": "north_dakota",
    "georgia": "north_carolina",
}  # pool -> area whose questions it answers
T = {}
for area in POOL_Q:
    for name, (tile, role) in area_tiles(cfg["areas"][area], cfg["grid"]).items():
        z = np.load(ROOT / "tiles" / f"{area}_{name}.npz")
        solar = z["solar"] if "solar" in z.files else None
        nwi = z["nwi"] if "nwi" in z.files else None
        valid = np.isfinite(z["ae"]).all(axis=0)
        if solar is not None:
            valid &= solar != SOLAR_LATER
        lon, lat = pixel_lonlat(tile)
        box = box_of(float(lon.mean()), float(lat.mean()), P["boxes"])
        iy, ix = pixel_cells(daily[box].lat.values, daily[box].lon.values, lat, lon)
        T[name] = {
            "area": area,
            "role": role,
            "valid": valid,
            "box": box,
            "iy": iy,
            "ix": ix,
            "cell": uni.index(box, iy, ix),
            "official": official_leaf_map(z["cdl"], LEAVES, CL, nwi=nwi, solar=solar),
        }
        if role == "test":
            T[name]["ae_map"] = np.load(ROOT / "maps" / f"ae_{name}.npz")["leaf"]
            T[name]["planted_map"] = np.load(ROOT / "maps" / f"planted_{name}.npz")["leaf"]
CTX, TILES = {}, {}
for area in POOL_Q:
    names = [n for n, t in T.items() if t["area"] == area]
    ds = daily[T[names[0]]["box"]]
    ac = np.zeros((ds.sizes["lat"], ds.sizes["lon"]), bool)
    for n in names:
        ac[T[n]["iy"], T[n]["ix"]] = True
    CTX[area] = AreaContext(
        leaves=LEAVES,
        groups=GROUPS,
        res_m=cfg["grid"]["res_m"],
        power=ds,
        area_cells=ac,
        pixel_cell={n: (T[n]["iy"], T[n]["ix"]) for n in names},
    )
    TILES[area] = {
        "train": [n for n in names if T[n]["role"] == "train"],
        "test": [n for n in names if T[n]["role"] == "test"],
    }


def answer(plan, pool, maps="official"):
    """Answer of a plan in a pool's test tiles, as one vector over their valid pixels (fixed tile order)."""
    tiles = TILES[pool]["test"]
    key = "official" if maps == "official" else f"{maps}_map"
    got = execute(plan, {n: T[n][key] for n in tiles}, CTX[pool])
    return np.concatenate([got[n][T[n]["valid"]] for n in tiles])


def question_text(q, wording, pool):
    text = q["question"] if wording != "reworded" else RW[q["id"]]
    if pool in ("minnesota", "georgia"):
        text = text.replace("North Dakota", "Minnesota").replace("North Carolina", "Georgia")
    return text


def n_extra(plan):
    return len(plan["near"]) + len(plan["climate"]) + int(plan["land"]["min_patch_ha"] is not None)


print(f"{len(T)} tiles loaded; pools {dict((a, TILES[a]['test']) for a in POOL_Q)}")

18 tiles loaded; pools {'north_dakota': ['Kidder', 'Foster', 'Eddy'], 'north_carolina': ['NC4', 'NC5', 'NC6'], 'minnesota': ['MN1', 'MN2', 'MN3'], 'georgia': ['GA1', 'GA2', 'GA3']}


In [2]:
from askmap.data import read_uspvdb
from askmap.layers import SOLAR_EXISTING
from askmap.models import fit_classifier, predict_map, training_sample
from askmap.score import median_ci
from askmap.tiles import solar_farm_ids, solar_raster

MC = cfg["classifier"]
OTHER = LEAVES.index("other")
LAST = cfg["uspvdb"]["last_year"]
pv = read_uspvdb(ROOT / "raw" / "uspvdbGeoJSON.zip", cfg["uspvdb"]["member"])
TR = {}
for area, acfg in cfg["areas"].items():
    for name, (tile, role) in area_tiles(acfg, cfg["grid"]).items():
        if role != "train":
            continue
        z = np.load(ROOT / "tiles" / f"{area}_{name}.npz")
        solar = z["solar"] if "solar" in z.files else None
        nwi = z["nwi"] if "nwi" in z.files else None
        valid = np.isfinite(z["ae"]).all(axis=0)
        if solar is not None:
            valid &= solar != SOLAR_LATER
        TR[name] = {
            "tile": tile,
            "solar": solar,
            "valid": valid,
            "leaf": official_leaf_map(z["cdl"], LEAVES, CL, nwi=nwi, solar=solar),
        }
# the same 10 farms as notebook 03 (same seed and rule)
rng = np.random.default_rng(MC["seed"])
farm_ids = sorted({i for n in TR for i in solar_farm_ids(pv, TR[n]["tile"], LAST)})
handful = sorted(rng.choice(farm_ids, size=MC["solar_handful"], replace=False).tolist())
keep = {}
for n, t in TR.items():
    k = t["valid"].copy()
    if t["solar"] is not None:
        hand = solar_raster(pv[pv.case_id.isin(handful)], t["tile"], LAST) == SOLAR_EXISTING
        k &= ~((t["solar"] == SOLAR_EXISTING) & ~hand)
    keep[n] = k


def onehot(leaf):
    return np.eye(len(LEAVES), dtype=np.float32)[leaf].transpose(2, 0, 1)


X, y = training_sample(
    [onehot(t["leaf"]) for t in TR.values()],
    [t["leaf"] for t in TR.values()],
    [keep[n] for n in TR],
    len(LEAVES),
    MC["per_class"],
    MC["seed"],
)
rf = fit_classifier(X, y, MC["trees"], MC["min_samples_leaf"], MC["seed"])
for pool in SEEN:
    for n in TILES[pool]["test"]:
        T[n]["clean_map"] = predict_map(rf, onehot(T[n]["official"]), OTHER)
print("handful:", handful)

handful: [400143, 400589, 400786, 401123, 401220, 401373, 401937, 402411, 403526, 404556]


In [3]:
rows = []
for pool in SEEN:
    for q in Q:
        if q["plan"]["area"] != pool:
            continue
        g = answer(q["plan"], pool)
        if g.sum() < SC["min_truth"]:
            continue
        rows.append(
            {
                "id": q["id"],
                "pool": pool,
                "F1_clean_planted": f1(answer(q["plan"], pool, "clean"), g),
                "F1_noisy_planted": f1(answer(q["plan"], pool, "planted"), g),
            }
        )
d = pd.DataFrame(rows)
res = {
    k: median_ci(d[k].to_numpy(), SC["bootstrap"], SC["seed"])
    for k in ("F1_clean_planted", "F1_noisy_planted")
}
(RES / "04b_d0_diagnostic.json").write_text(json.dumps({"post_hoc": True, **res, "n": len(d)}, indent=1))
print(res)
d.sort_values("F1_clean_planted").round(3).head(12)

{'F1_clean_planted': (1.0, 1.0, 1.0), 'F1_noisy_planted': (0.7765921643933245, 0.6346409799454082, 0.9179598295362595)}


,id,pool,F1_clean_planted,F1_noisy_planted
0,nd01,north_dakota,1.0,0.636
26,nd29,north_dakota,1.0,0.951
27,nd30,north_dakota,1.0,0.525
28,nd31,north_dakota,1.0,0.613
29,nd32,north_dakota,1.0,0.449
30,nd33,north_dakota,1.0,0.629
31,nd34,north_dakota,1.0,0.259
32,nd35,north_dakota,1.0,0.738
33,nd36,north_dakota,1.0,0.944
34,nd37,north_dakota,1.0,0.952
